In [ ]:
import os

In [ ]:
from  dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
keyvalue = os.getenv("KEYNAME")
print(f"keyvalue => {keyvalue}")

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm_model = ChatGroq(model="qwen/qwen3-32b")

In [ ]:
llm_model.invoke("what is latest trend in AI field today")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader,TextLoader, Docx2txtLoader

In [ ]:
from pathlib import Path

In [ ]:
file_directory = Path("./data/multi_chat")

In [ ]:
all_document=[]
for file_name in file_directory.iterdir():
    if not file_name.is_file():
        continue
    file_extension = (Path(file_name.name).suffix.lower())
    if file_extension == ".docx":
        loader = Docx2txtLoader(file_name)
    elif file_extension == ".pdf":
        loader = PyPDFLoader(file_name)
    elif file_extension == ".txt":
        loader = TextLoader(file_name)
    else:
        continue

    all_document.extend(loader.load())



        
    

In [ ]:
print(all_document)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=300)
chunks = splitter.split_documents(all_document)

In [ ]:
from langchain_community.vectorstores import FAISS
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.embeddings import OpenAIEmbeddings

In [ ]:
# embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
# embeddings = OllamaEmbeddings(model="mxbai-embed-large")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
	


In [ ]:
FAISS_INDEX_PATH = Path("faiss_index")

In [ ]:
if FAISS_INDEX_PATH.exists():
    vector_store = FAISS.load_local(folder_path=str(FAISS_INDEX_PATH),embeddings=embeddings, allow_dangerous_deserialization=True)
    print("retrieved faiss index")
else:
    vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)
    # vectors = [embeddings.embed_query(doc.page_content) for doc in chunks]
    # vector_store = FAISS.from_embeddings(embeddings=vectors, documents=chunks)
    vector_store.save_local(str("faiss_index"))
    print("created faiss index")

In [ ]:
# retriever = vector.as_retriever()
retriever =vector_store.as_retriever()

In [ ]:
from langchain_core.prompts import  ChatPromptTemplate, MessagesPlaceholder

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
("system",(
    "You are an assistant designed to provide answer using the provided context "
    "Relay your answer only on the retrieved information.  If you don't have answer then "
    "reply that you don't have information. Keep you answer more concise and no longer three sentences. \n\n {context}"
)),
MessagesPlaceholder("chat_history_test"),
("human","{input}")
])


In [ ]:
# Prompt for contextual question rewriting
contextualize_the_question_prompt = ChatPromptTemplate.from_messages([
("system",(
    "Given a conversational history and recent user query, rewrite the user question as standalone "
    "question which doesn't require previous context.  Don't answer the user question and rewrite the question, if required, "
    " otherwise return the query without change"
)),

MessagesPlaceholder("chat_history_test"),
("human","{input}")

])

In [ ]:
from langchain.chains import create_history_aware_retriever, create_retrieval_chain

In [ ]:
history_aware_retriver = create_history_aware_retriever(llm_model,retriever , contextualize_the_question_prompt) 

In [ ]:
from langchain.chains.combine_documents import create_stuff_documents_chain

In [ ]:
qa_chain = create_stuff_documents_chain(llm_model,qa_prompt)

In [ ]:
rag_chain = create_retrieval_chain(
    history_aware_retriver,
    qa_chain
)

In [ ]:
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

import streamlit as st

In [ ]:
def _get_message_history(session_id:str) -> BaseChatMessageHistory:
    if "store" not in st.session_state:
        st.session_state.store={}
    if session_id not in st.session_state.store:
        st.session_state.store[session_id]=ChatMessageHistory()
    return st.session_state.store[session_id]


In [ ]:
session_id = "test_conversational_rag"

1) 🧩 What history_messages_key="chat_history_test" Means
    This tells LangChain:
    “When building the prompt, insert the messages from _get_message_history() into the placeholder called chat_history_test.”

2) {input}, {context}, chat_history — these are not reserved keywords. You define them, and LangChain matches them based on:
    Prompt template
    Chain configuration
    Message history manager

In [ ]:
chain = RunnableWithMessageHistory(
    rag_chain,
    _get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history_test",
    output_messages_key="answer"
)

In [ ]:
# user_input="what is the main topic of the document?"
# user_input ="what is President Zelenskyy said in their speech in parliament?"
user_input ="customer termination ?"

In [ ]:
from typing import Optional, List
from langchain_core.messages import BaseMessage

In [ ]:
# chat_history:Optional[List[BaseMessage]]=None

response = chain.invoke(
{"input":user_input},
config={"configurable":{"session_id":session_id}}
)

In [ ]:
answer = response.get("answer","No answer.")

In [ ]:
answer